# dr_evt 07: a market over dr_evt clusters, the design and a run

## The problem

A federation has several clusters. Users submit jobs and would rather run on some
clusters than others: one is faster, one is cheaper, one has the right accelerator. A
job may also be composite, several legs that must start together on possibly different
clusters. Someone has to decide, for every job, which cluster runs it and what it pays.

dr_evt simulates one cluster faithfully: a queue, EASY backfilling, node accounting,
start and end times per job. This notebook is about the layer above it: an auction that
decides placement across clusters and then lets each cluster's own scheduler do its job.
It walks through the design decisions in the order they were made, what each one
produced in code, and ends with a complete run.

Five things were fixed at the start and shaped everything after:

1. dr_evt stays the truth about scheduling. Nothing about queues, backfilling or node
   accounting is reimplemented; the auction is a client of dr_evt.
2. The client must not look inside a cluster. It may ask what is free, submit, and read
   back what happened. That keeps the auction independent of how a cluster schedules.
3. A cluster running in the same process and a cluster served over gRPC must behave
   identically, so the same experiment can run on a laptop or across machines.
4. Everything is deterministic and hashed, so a run can be reproduced and compared.
5. The mechanism that decides is a plug: VCG first, as the truthful reference, and a
   learned mechanism next to it.

Sections:

1. Decision 1: the boundary between the market and a cluster
2. Decision 2: what dr_evt had to gain, and why
3. Decision 3: one contract, two transports, wrapping what dr_evt provides
4. Decision 4: what is sold, and what a bid is
5. Decision 5: the mechanism as a plug, VCG first
6. Decision 6: windows, the loop, and the outputs
7. The run: input files, one window by hand, the whole run, gRPC, the command line
8. RegretFormer, the learned mechanism

In [1]:
from pathlib import Path
import inspect, os, sys, tempfile, subprocess, json, re
import pandas as pd
from IPython.display import Markdown, display

DR_EVT = Path.cwd().resolve()                        # run from learn/ or from the repository root
while not (DR_EVT / "CMakeLists.txt").exists() and DR_EVT != DR_EVT.parent:
    DR_EVT = DR_EVT.parent
assert (DR_EVT / "CMakeLists.txt").exists(), "run this notebook from learn/ inside a dr_evt checkout"
INSTALL = Path(os.environ.get("DR_EVT_INSTALL", DR_EVT / "install"))   # the cmake install prefix
OUT = Path.cwd() / "output"                                            # scratch space, gitignored
OUT.mkdir(exist_ok=True)
sys.path[:0] = [str(INSTALL / "lib" / "python"), str(DR_EVT / "python")]   # dr_evt extension, then the package

import dr_evt_market as m
from dr_evt_market.platforms import base as platform_base
from dr_evt_market.platforms import inprocess, grpc as grpc_adapter, server
from dr_evt_market.mechanisms import base as mechanism_base, clearing, vcg
from dr_evt_market import controller, inputs

def show(obj):
    """Display the source of a class or function."""
    display(Markdown(f"```python\n{inspect.getsource(obj).rstrip()}\n```"))

def show_file(relative, start_pattern, lines, language="cpp"):
    """Display a slice of a repository file starting at the first line matching a pattern."""
    text = (DR_EVT / relative).read_text().splitlines()
    start = next(i for i, line in enumerate(text) if re.search(start_pattern, line))
    snippet = "\n".join(f"{start + 1 + k:4d}  {line}" for k, line in enumerate(text[start:start + lines]))
    display(Markdown(f"`{relative}:{start + 1}`\n\n```{language}\n{snippet}\n```"))

EXAMPLES = DR_EVT / "python" / "examples" / "market"
print("dr_evt_market", m.__version__)

dr_evt_market 0.1.0


## 1. Decision 1: the boundary between the market and a cluster

The first question was what the auction actually needs from a cluster. The answer turned
out to be small: how many nodes are free right now; accept these jobs now and give me a
handle for each; tell me later when each one started and ended. Everything else, the
queue, the backfilling, the node accounting, stays inside dr_evt. Writing that answer
down as an interface is what makes the rest possible: the market code never sees a
`dr_evt.Simulation` or a gRPC stub, and a cluster is anything that implements the
interface.

The interface lives in `python/dr_evt_market/platforms/base.py`: five frozen records,
four error types, and a protocol of six methods. Two smaller decisions are baked into
it.

Times are integers on the way in. dr_evt keeps the fractional part of a timestamp as a
single-precision float, so a job appended at 10.123 does not start on
`advance_to(10.123)`. The market therefore keeps a whole-second clock and rejects
anything else at the boundary.

The records are frozen. A snapshot or a timing record is evidence of what a cluster
said at one moment; nothing downstream may edit it.

### `SubmitRequest`: what the market hands to a cluster

One leg of one job, exactly as the cluster needs it. Notice what is absent: no bid, no

| field | type | meaning |
|---|---|---|
| `key` | str | the market's own name for the leg, `job_id/leg_id`, so a timing record can be joined back to the decision that produced it |
| `submit_s` | int | the submission time; the market always submits at the window time |
| `num_nodes` | int | node demand |
| `limit_s` | int | the requested time limit; dr_evt reserves nodes against it |
| `q_id` | str | dr_evt's numeric queue id, `"1"` by default; the scheduler does not read it |

In [2]:
show(platform_base.SubmitRequest)

```python
@dataclass(frozen=True)
class SubmitRequest:
    """Describe one client-owned job or composite leg to submit."""

    key: str
    submit_s: int
    num_nodes: int
    limit_s: int
    q_id: str = "1"
```

### `JobTiming`: the receipt

What a cluster says happened to one leg. The market does not use it to decide anything;
by the time it exists the decision is made. It is read for two reasons: to confirm that
the cluster did what the auction assumed, that a winner handed to the cluster it won
did start at the window, and to record per-job outcomes, wait and turnaround, for the
evaluation.

| field | type | meaning |
|---|---|---|
| `handle` | int | dr_evt's job index, returned by `submit`; permanent, never reused |
| `key` | str | the `SubmitRequest.key` the market gave |
| `submit_s` | float | the submission time dr_evt recorded |
| `begin_s`, `end_s` | float | start and end as dr_evt scheduled them; -1 until the job has started |
| `limit_s` | int | the time limit |
| `actual_run_s` | float | the run time dr_evt applied; equal to the limit in limit mode |
| `num_nodes` | int | node demand |
| `scheduled` | bool | whether the job has started; a job that never fits stays `False` |

In [3]:
show(platform_base.JobTiming)

```python
@dataclass(frozen=True)
class JobTiming:
    """Mirror one DR_EVT job timing record in seconds."""

    handle: int
    key: str
    submit_s: float
    begin_s: float
    end_s: float
    limit_s: int
    actual_run_s: float
    num_nodes: int
    scheduled: bool
```

### `PlatformSnapshot`: the supply

What the auction reads before it decides. `free_nodes` is the supply it sells;
`waiting_jobs` lets it confirm nothing is queued behind its own submissions;
`current_utilization` is a diagnostic. A `None` means the adapter cannot know, never
zero.

| field | type | meaning |
|---|---|---|
| `name` | str | the cluster's name |
| `time_s` | int | the cluster clock at the snapshot |
| `total_nodes` | int | the cluster's size |
| `free_nodes` | int | nodes not in use right now |
| `in_use_nodes` | int | nodes held by running jobs |
| `waiting_jobs` | int | jobs in the cluster's own queue |
| `current_utilization` | float or None | in-use over total at this instant |

In [4]:
show(platform_base.PlatformSnapshot)

```python
@dataclass(frozen=True)
class PlatformSnapshot:
    """Capture common scheduler capacity, queue, and utilization state."""

    name: str
    time_s: int
    total_nodes: int
    free_nodes: int
    in_use_nodes: int
    waiting_jobs: int
    current_utilization: float | None = None
```

### `PlatformReport`: what a cluster hands back at the end

Every timing record, dr_evt's statistics, and the paths of dr_evt's own output files,
so the run can be joined to decisions and the cluster's files are kept as they are.

| field | type | meaning |
|---|---|---|
| `name` | str | the cluster's name |
| `timings` | tuple of `JobTiming` | one record per submitted leg, in handle order |
| `statistics` | dict | dr_evt's `Statistics`: completed jobs, makespan, utilization, average wait and turnaround |
| `simulated_trace_path` | str | dr_evt's simulated trace CSV |
| `resource_trace_path` | str | dr_evt's resource history CSV, occupancy over time |

In [5]:
show(platform_base.PlatformReport)

```python
@dataclass(frozen=True)
class PlatformReport:
    """Collect final timing records, statistics, and output paths."""

    name: str
    timings: tuple[JobTiming, ...]
    statistics: dict[str, float]
    simulated_trace_path: str | None
    resource_trace_path: str | None
```

### The errors, and why the boundary refuses things

dr_evt is permissive in ways a market cannot afford. It accepts a job larger than the
cluster and drops it silently. It does not check that submission times are
non-decreasing. It truncates fractional seconds. Each of those would make a market run
wrong without any error. So the boundary refuses them, and it raises one of four types
so the caller can react by meaning instead of parsing a message.

| error | base | raised for |
|---|---|---|
| `ClockViolation` | `ValueError` | a fractional, backward or out-of-order time |
| `StructuralRejection` | `ValueError` | a leg that can never fit, or a non-positive size or limit |
| `ConfigurationError` | `ValueError` | a bad queue id, policy, name or address |
| `InfrastructureFailure` | `RuntimeError` | a cluster, stream or process that failed, or a call after `finish()` |

`validate` checks the shape of a request on its own; the checks that need a cluster's
clock or size live in the adapters.

In [6]:
show(platform_base.validate)

```python
def validate(request: SubmitRequest) -> None:
    """Validate the context-independent shape of one submission request."""

    if not _is_integer(request.submit_s):
        raise ClockViolation("submit_s must be an integer number of seconds")
    if not _is_integer(request.num_nodes) or request.num_nodes < 1:
        raise StructuralRejection("num_nodes must be a positive integer")
    if not _is_integer(request.limit_s) or request.limit_s < 1:
        raise StructuralRejection("limit_s must be a positive integer")
    if (not isinstance(request.q_id, str) or not request.q_id.isdigit()
            or not 1 <= int(request.q_id) <= 10):
        raise ConfigurationError("q_id must be a digit string in 1..10")
```

### `PlatformSession`: the six methods

A `typing.Protocol` rather than a base class: any object with these members is a
cluster, and one test suite runs against every implementation.

| member | meaning |
|---|---|
| `name`, `total_nodes` | identity and size |
| `now()` | the cluster clock |
| `submit(jobs)` | validate a batch, append it in one call, return one handle per leg; appending only queues |
| `advance_to(time_s)` | process every event up to that time, including the queue evaluation at that time |
| `snapshot()` | the `PlatformSnapshot` at the current time, taken after `advance_to` |
| `timings(handles)` | receipts in the requested order |
| `finish()` | drain, write both traces, close the cluster, return the `PlatformReport` |

In [7]:
show(platform_base.PlatformSession)

```python
class PlatformSession(Protocol):
    """Define the common lifecycle of one simulated platform session."""

    name: str
    total_nodes: int

    def now(self) -> int:
        """Return the adapter's current integer simulation time."""
        ...

    def submit(self, jobs: Sequence[SubmitRequest]) -> list[int]:
        """Validate and submit a batch, returning DR_EVT handles in order."""
        ...

    def advance_to(self, time_s: int) -> None:
        """Advance through all scheduler events at or before time_s."""
        ...

    def snapshot(self) -> PlatformSnapshot:
        """Return one consistent snapshot of current scheduler state.

        Call after advance_to(); between submit and advance the queue has not
        yet been evaluated by the scheduler.
        """
        ...

    def timings(self, handles: Sequence[int]) -> list[JobTiming]:
        """Return timing records for the requested handles in order."""
        ...

    def finish(self) -> PlatformReport:
        """Drain submitted work and return final records and output paths."""
        ...
```

## 2. Decision 2: what dr_evt had to gain, and why

Most of the boundary maps onto what dr_evt's streaming API already offered: append jobs,
advance the clock, read free nodes. One thing was missing. The only way to learn when a
job started was to finish the run and parse the session CSV, and the market needs the
receipt after every window, while the run is live, on both transports. So dr_evt gained
a per-job accessor: a `Job_Timing` record, `get_job_timing` for one handle and
`get_job_timings` for many. Times are seconds as doubles, -1 until the job starts; an
unknown or reclaimed handle raises `std::out_of_range`, which becomes `IndexError` in
Python and the boundary's `KeyError`.

In [8]:
show_file("src/sim/sim.hpp", r"struct Job_Timing", 20)
show_file("src/sim/sim.cpp", r"::get_job_timing\(", 32)

`src/sim/sim.hpp:429`

```cpp
 429    struct Job_Timing {
 430      job_no_t job_idx;
 431      sim_time_t submit_time;      ///< -1 when the record has no valid submission.
 432      sim_time_t begin_time;       ///< -1 until the job has started.
 433      sim_time_t end_time;         ///< -1 until the job has started.
 434      timeout_t limit_time;
 435      tdiff_t actual_run_time;     ///< 0 until determined.
 436      num_nodes_t num_nodes;
 437      bool scheduled;              ///< Job_Record::is_scheduled().
 438    };
 439  
 440    /**
 441     * @brief Return read-only timing for one job.
 442     * @details Throws std::out_of_range with the offending job identifier when
 443     * the job was never appended or was already reclaimed by capacity pressure
 444     * at append time or by flush_completed_jobs(). A caller that needs every
 445     * job's timing must read it before the next append or flush. For a job that
 446     * has started, end_time is its projected end, equal to start plus actual
 447     * run time, which is how dr_evt records job ends.
 448     * @param[in] job_idx Permanent job identifier.
```

`src/sim/sim.cpp:944`

```cpp
 944  BasicSimulation<TraceType>::get_job_timing(job_no_t job_idx) const {
 945    const size_t num_reclaimed = m_trace.num_reclaimed();
 946    const bool is_reclaimed = job_idx < num_reclaimed;
 947    const size_t resident_idx =
 948        is_reclaimed ? static_cast<size_t>(0) : job_idx - num_reclaimed;
 949    if (is_reclaimed || resident_idx >= m_trace.data().size()) {
 950      throw std::out_of_range("get_job_timing(): job_idx=" +
 951                              std::to_string(job_idx) +
 952                              " is not available");
 953    }
 954  
 955    const auto &job = m_trace.job_at(job_idx);
 956    const bool scheduled = job.is_scheduled();
 957    const epoch_t unscheduled = Job_Record::unscheduled_sentinel();
 958    const epoch_t submit_epoch = job.get_submit_time();
 959  
 960    const sim_time_t submit_time =
 961        submit_epoch == unscheduled
 962            ? static_cast<sim_time_t>(-1.0)
 963            : convert_epoch<sim_time_t>(submit_epoch);
 964    const sim_time_t begin_time =
 965        scheduled ? convert_epoch<sim_time_t>(job.get_begin_time())
 966                  : static_cast<sim_time_t>(-1.0);
 967    const sim_time_t end_time =
 968        scheduled ? convert_epoch<sim_time_t>(job.get_end_time())
 969                  : static_cast<sim_time_t>(-1.0);
 970  
 971    return {job_idx, submit_time, begin_time, end_time, job.get_limit_time(),
 972            job.get_actual_run_time(), job.get_num_nodes(), scheduled};
 973  }
 974  
 975  template <typename TraceType>
```

The pybind11 binding exposes the record and both accessors, which is what the in-process
adapter calls. The same call went onto the gRPC session stream as `GetJobTimings`,
message fields 18 and 19, so a remote cluster gives the same receipt as a local one.

In [9]:
show_file("python/dr_evt_bindings.cpp", r"py::class_<Simulation::Job_Timing>", 30)
show_file("src/proto/dr_evt_service.proto", r"message GetJobTimingsRequest", 6, language="protobuf")
show_file("src/proto/dr_evt_service.proto", r"message JobTimingData", 16, language="protobuf")
show_file("src/proto/dr_evt_server.cpp", r"case ClientMessage::kGetJobTimings", 23)

`python/dr_evt_bindings.cpp:219`

```cpp
 219    py::class_<Simulation::Job_Timing>(m, "JobTiming")
 220        .def_readonly("job_idx", &Simulation::Job_Timing::job_idx,
 221                      "int: Permanent job identifier.")
 222        .def_readonly("submit_time", &Simulation::Job_Timing::submit_time,
 223                      "float: Submission time, or -1 when invalid.")
 224        .def_readonly("begin_time", &Simulation::Job_Timing::begin_time,
 225                      "float: Projected start time, or -1 until the job starts.")
 226        .def_readonly("end_time", &Simulation::Job_Timing::end_time,
 227                      "float: Projected end time, or -1 until the job starts.")
 228        .def_readonly("limit_time", &Simulation::Job_Timing::limit_time,
 229                      "int: Requested wall-time limit in seconds.")
 230        .def_readonly("actual_run_time",
 231                      &Simulation::Job_Timing::actual_run_time,
 232                      "float: Actual or scheduled run duration.")
 233        .def_readonly("num_nodes", &Simulation::Job_Timing::num_nodes,
 234                      "int: Requested node count.")
 235        .def_readonly("scheduled", &Simulation::Job_Timing::scheduled,
 236                      "bool: Whether the job has scheduled timing.")
 237        .def("__repr__", [](const Simulation::Job_Timing &timing) {
 238          return "JobTiming(job_idx=" + std::to_string(timing.job_idx) +
 239                 ", submit_time=" + std::to_string(timing.submit_time) +
 240                 ", begin_time=" + std::to_string(timing.begin_time) +
 241                 ", end_time=" + std::to_string(timing.end_time) +
 242                 ", limit_time=" + std::to_string(timing.limit_time) +
 243                 ", actual_run_time=" +
 244                 std::to_string(timing.actual_run_time) +
 245                 ", num_nodes=" + std::to_string(timing.num_nodes) +
 246                 ", scheduled=" + (timing.scheduled ? "true" : "false") +
 247                 ")";
 248        });
```

`src/proto/dr_evt_service.proto:160`

```protobuf
 160  message GetJobTimingsRequest {
 161      repeated uint64 job_idx = 1;
 162  }
 163  message GetStatisticsRequest {}
 164  message GetTraceSizeRequest {}
 165  
```

`src/proto/dr_evt_service.proto:270`

```protobuf
 270  message JobTimingData {
 271      uint64 job_idx = 1;
 272      double submit_time = 2;
 273      double begin_time = 3;
 274      double end_time = 4;
 275      double limit_time = 5;
 276      double actual_run_time = 6;
 277      uint32 num_nodes = 7;
 278      bool scheduled = 8;
 279  }
 280  
 281  // Timing snapshots are returned in the same order as the requested IDs.
 282  message GetJobTimingsResponse {
 283      repeated JobTimingData timings = 1;
 284  }
 285  
```

`src/proto/dr_evt_server.cpp:381`

```cpp
 381          case ClientMessage::kGetJobTimings: {
 382            require_init(sim);
 383            const auto &request = req.get_job_timings();
 384            std::vector<dr_evt::job_no_t> job_idxs;
 385            job_idxs.reserve(request.job_idx_size());
 386            for (const uint64_t job_idx : request.job_idx()) {
 387              job_idxs.push_back(static_cast<dr_evt::job_no_t>(job_idx));
 388            }
 389            const auto timings = sim->get_job_timings(job_idxs);
 390            auto *out = resp.mutable_get_job_timings();
 391            for (const auto &timing : timings) {
 392              auto *data = out->add_timings();
 393              data->set_job_idx(static_cast<uint64_t>(timing.job_idx));
 394              data->set_submit_time(timing.submit_time);
 395              data->set_begin_time(timing.begin_time);
 396              data->set_end_time(timing.end_time);
 397              data->set_limit_time(static_cast<double>(timing.limit_time));
 398              data->set_actual_run_time(timing.actual_run_time);
 399              data->set_num_nodes(timing.num_nodes);
 400              data->set_scheduled(timing.scheduled);
 401            }
 402            break;
 403          }
```

The `PlatformReport` carries dr_evt's two output files and its precision. Setting the
output path, the resource trace and millisecond output from Python, and writing the
resource history at `finish()`, needed a few more `SimParams` fields and
`write_resource_trace(filename)` in the binding. Two fields were left unbound on
purpose: `max_time`, which the simulation does not read, and the job-store settings,
which apply only inside batch `run()`.

In [10]:
show_file("python/dr_evt_bindings.cpp", r'"msec_output"', 36)
show_file("python/dr_evt_bindings.cpp", r'"write_resource_trace"', 6)

`python/dr_evt_bindings.cpp:100`

```cpp
 100        .def_readwrite("msec_output", &Sim_Params::m_msec_output,
 101                       "bool: Emit output timestamps with three decimal places.")
 102        .def_readwrite("run_time_scale", &Sim_Params::m_run_time_scale,
 103                       "float: Sampler scale as a fraction of time_limit: the "
 104                       "NORMAL mean, the LOGNORMAL median and the UNIFORM "
 105                       "lower bound are limit times scale.")
 106        .def_readwrite("run_time_stddev", &Sim_Params::m_run_time_stddev,
 107                       "float: Sampler spread: NORMAL standard deviation is "
 108                       "limit times stddev, LOGNORMAL sigma is stddev, "
 109                       "UNIFORM upper bound is limit times (scale plus "
 110                       "stddev). NORMAL and LOGNORMAL samples are capped at "
 111                       "time_limit; UNIFORM is not.")
 112        .def_readwrite(
 113            "run_time_distribution", &Sim_Params::m_run_time_distribution,
 114            "DistributionType: Distribution used for trace-loaded jobs when "
 115            "run_time_mode is DISTRIBUTION.")
 116        .def_property(
 117            "outfile", [](const Sim_Params &params) {
 118              return params.get_outfile();
 119            },
 120            [](Sim_Params &params, const std::string &outfile) {
 121              params.set_outfile(outfile);
 122            },
 123            "str: Simulated-trace output filename. An empty string selects the "
 124            "default derived from infile.")
 125        .def_property(
 126            "resource_trace", [](const Sim_Params &params) {
 127              return params.get_resource_trace();
 128            },
 129            [](Sim_Params &params, const std::string &resource_trace) {
 130              params.set_resource_trace(resource_trace);
 131            },
 132            "str: Optional resource-history output filename.");
 133  
 134    // One value passed to Simulation.append_jobs().
 135    py::class_<Simulation::Job_Append_Request>(m, "JobAppendRequest")
```

`python/dr_evt_bindings.cpp:364`

```cpp
 364        .def("write_resource_trace", &Simulation::write_resource_trace,
 365             py::arg("filename"),
 366             "Write the resource-history CSV (time,free_nodes,allocated_nodes) "
 367             "to filename. The batch CLI and the gRPC server call this at the "
 368             "end of a run; from Python pass params.resource_trace or any "
 369             "path.")
```

That is the whole footprint in dr_evt: one record, two accessors, one gRPC message, and
the bindings, each with its tests. The scheduler itself is unchanged.

## 3. Decision 3: one contract, two transports, wrapping what dr_evt provides

dr_evt already ships both halves of a cluster, and neither was changed.

The platform is `dr_evt.Simulation`, the pybind11 class over `BasicSimulation<Trace>`.
It is driven the way `python/example_streaming.py` drives it: fill a `SimParams`,
construct, `append_job` or `append_jobs`, `advance_to`, then read `get_available_nodes`,
`get_nodes_in_use`, `get_active_job_count` and `get_statistics`.

The client and server are `dr_evt_server` and `python/grpc_multi_server.py`. The server
owns one `Simulation` per `Session` RPC and answers requests serially on a
bidirectional stream of `ClientMessage` and `ServerMessage`, each carrying a
`request_id` and a `oneof` with the request kind. The client's `load_stubs` runs
`protoc` at run time, so nothing generated is checked in, and `ServerSession.call`
sends one request, reads one response, and checks that the ids match.

In [11]:
show_file("python/example_streaming.py", r"params = dr_evt.SimParams", 30, language="python")
show_file("src/proto/dr_evt_service.proto", r"^service SimulationService", 3, language="protobuf")
show_file("src/proto/dr_evt_service.proto", r"^message ClientMessage", 24, language="protobuf")
show_file("src/proto/dr_evt_server.cpp", r"Concrete gRPC service that owns one Simulation", 20)
show_file("python/grpc_multi_server.py", r"^class ServerSession", 36, language="python")

`python/example_streaming.py:25`

```python
  25      params = dr_evt.SimParams()
  26      # Resolve relative to this script's own location, not the caller's
  27      # cwd - a bare "examples/sample_trace.csv" only resolves correctly
  28      # if invoked as `python example_streaming.py` from within python/
  29      # itself; `python3 python/example_streaming.py` from the repo root
  30      # (an equally natural way to run it - and how CI's own test now
  31      # invokes it) would otherwise fail with "Failed to initialize data
  32      # columns" since that path doesn't exist relative to the repo root.
  33      params.infile = os.path.join(
  34          os.path.dirname(os.path.abspath(__file__)),
  35          "examples", "sample_trace.csv")
  36      params.total_nodes = 100
  37      params.trace_format = "simple"
  38      params.timestamp_format = "epoch"
  39      params.run_time_mode = dr_evt.RunTimeMode.LIMIT
  40      params.backfill_policy = dr_evt.BackfillPolicy.EASY
  41      params.priority_policy = dr_evt.PriorityPolicy.FCFS
  42      params.verbose = False
  43  
  44      # Read external job requests. append_job() is the public streaming API;
  45      # initialize_trace() loads a batch trace and is not a source of jobs to
  46      # re-submit individually.
  47      with open(params.infile, newline='', encoding='utf-8') as trace:
  48          jobs = list(csv.DictReader(trace))[:10]
  49  
  50      # Create simulator.
  51      sim = dr_evt.Simulation(params)
  52      print(f"Read {len(jobs)} jobs from trace")
  53  
  54      print("\n" + "="*60)
```

`src/proto/dr_evt_service.proto:21`

```protobuf
  21  service SimulationService {
  22      rpc Session(stream ClientMessage) returns (stream ServerMessage);
  23  }
```

`src/proto/dr_evt_service.proto:27`

```protobuf
  27  message ClientMessage {
  28      // Echoed back on the matching ServerMessage so the client can correlate
  29      // out-of-order or interleaved responses. Not interpreted by the server.
  30      uint64 request_id = 1;
  31      oneof request {
  32          InitRequest init = 2;
  33          InitializeTraceRequest initialize_trace = 3;
  34          AdvanceToRequest advance_to = 4;
  35          RunUntilExclusiveRequest run_until_exclusive = 5;
  36          GetCurrentTimeRequest get_current_time = 6;
  37          GetNodesInUseRequest get_nodes_in_use = 7;
  38          GetAvailableNodesRequest get_available_nodes = 8;
  39          GetActiveJobCountRequest get_active_job_count = 9;
  40          GetFCFSHeadShadowTimeRequest get_fcfs_head_shadow_time = 10;
  41          GetStatisticsRequest get_statistics = 11;
  42          GetTraceSizeRequest get_trace_size = 12;
  43          AppendJobRequest append_job = 13;
  44          AppendJobsRequest append_jobs = 14;
  45          FinishSimulationRequest finish_simulation = 15;
  46          GetBackfillWindowRequest get_backfill_window = 16;
  47          GetCurrentUtilizationRequest get_current_utilization = 17;
  48          GetJobTimingsRequest get_job_timings = 18;
  49      }
  50  }
```

`src/proto/dr_evt_server.cpp:128`

```cpp
 128   * @brief Concrete gRPC service that owns one Simulation per Session RPC.
 129   * @details Session state is deliberately local to Session(), isolating
 130   * concurrent clients. Requests on an individual stream are processed in read
 131   * order and produce exactly one correlated response unless transport writing
 132   * fails or the peer cancels the RPC.
 133   */
 134  class SimulationServiceImpl final : public SimulationService::Service {
 135  public:
 136    /**
 137     * @brief Serve one bidirectional streaming simulation session.
 138     * @param[in] context gRPC server context for cancellation and metadata.
 139     * @param[in,out] stream Bidirectional client-request/server-response stream.
 140     * @return gRPC OK on normal completion, otherwise an RPC status error.
 141     * @details The session initializes one Simulation and then processes
 142     * append, advance, query, and output requests serially.
 143     */
 144    Status
 145    Session(ServerContext *context,
 146            ServerReaderWriter<ServerMessage, ClientMessage> *stream) override {
 147      // sim_params must outlive sim: Simulation stores its parameters
```

`python/grpc_multi_server.py:67`

```python
  67  class ServerSession:
  68      """A synchronous request/response wrapper over one Session stream."""
  69  
  70      def __init__(self, address, grpc, messages, service):
  71          self.address = address
  72          self.messages = messages
  73          self._outgoing = queue.Queue()
  74          self._request_id = 1
  75          self._channel = grpc.insecure_channel(address)
  76          self._responses = service.SimulationServiceStub(self._channel).Session(
  77              self._request_iterator())
  78  
  79      def _request_iterator(self):
  80          while True:
  81              request = self._outgoing.get()
  82              if request is None:
  83                  return
  84              yield request
  85  
  86      def call(self, request):
  87          request.request_id = self._request_id
  88          self._request_id += 1
  89          self._outgoing.put(request)
  90          try:
  91              response = next(self._responses)
  92          except StopIteration as error:
  93              raise RuntimeError(f"{self.address}: server closed the stream") from error
  94          if response.request_id != request.request_id:
  95              raise RuntimeError(
  96                  f"{self.address}: response id {response.request_id} does not match "
  97                  f"request id {request.request_id}"
  98              )
  99          if response.HasField("error"):
 100              raise RuntimeError(f"{self.address}: {response.error.message}")
 101          return response
 102  
```

The market wraps each of them behind the contract of section 1, so the market never
sees a `Simulation` or a stub, and an experiment runs on a laptop with every cluster in
one process or with clusters on other machines without a change. The wrapping is thin
on purpose: every contract method is one or two dr_evt calls. Here is the mapping.

| contract | in process, `dr_evt.Simulation` | over gRPC, `ClientMessage` to `dr_evt_server` |
|---|---|---|
| construction | the `SimParams` of `example_streaming.py` plus `msec_output`, `outfile` and `resource_trace`, then `Simulation(params)`; a header-only input CSV, because dr_evt parses the header even when streaming | the `InitRequest` of `grpc_multi_server.py` plus `msec_output` and a `session_name`; the header file goes into a directory the server can read |
| `now()` | `get_current_time()` | `get_current_time` |
| `submit(jobs)` | one `append_jobs` of `JobAppendRequest`; the returned job indices are the handles | one `append_jobs` of `JobAppendData`; the returned job indices are the handles |
| `advance_to(t)` | `advance_to(t)` | `advance_to` |
| `snapshot()` | `get_available_nodes`, `get_nodes_in_use`, `get_active_job_count`, `get_current_utilization` | `get_statistics` for available, in use, waiting and time, and `get_current_utilization` |
| `timings(handles)` | `get_job_timings(handles)`, the accessor of section 2 | `get_job_timings`, field 18, the message of section 2 |
| `finish()` | `write_simulated_trace`, `write_resource_trace` (bound in section 2), `get_statistics` | `finish_simulation`, which writes both files on the server and returns their names and the statistics |

What the wrappers add is the same on both sides: the checks dr_evt does not make (a leg
that fits in `total_nodes`, an integer time that is not before the clock, submission
times that do not decrease across batches), a ledger from handles to the market's keys,
the mapping of every failure onto the four error types, and a guard that refuses calls
after `finish()`. A test holds both transports to identical timing records and trace
rows.

`InProcessPlatform(name, total_nodes, work_dir)` keeps the `SimParams` object alive as
long as the simulation, which holds it by reference, and writes the header file into
`work_dir`.

| method | what it does |
|---|---|
| `submit` | `validate`, then the cluster's own checks; then one `append_jobs` call; the ledger maps handles to keys |
| `advance_to` | rejects a non-integer or backward time, then calls dr_evt |
| `snapshot` | free, in-use and waiting counts and utilization now |
| `timings` | refuses handles it never issued, maps dr_evt's `IndexError` to `KeyError`, attaches the key |
| `finish` | advances far enough to drain, reads every receipt, writes both traces, caches the report, refuses further calls |

In [12]:
show(inprocess.InProcessPlatform.submit)
show(inprocess.InProcessPlatform.advance_to)
show(inprocess.InProcessPlatform.snapshot)
show(inprocess.InProcessPlatform.timings)
show(inprocess.InProcessPlatform.finish)

```python
    def submit(self, jobs: Sequence[SubmitRequest]) -> list[int]:
        """Validate and append one chronologically ordered request batch."""
        self._ensure_open()
        requests = list(jobs)
        previous_submit = self._last_submit_s
        current_time = self.now()
        for request in requests:
            validate(request)
            if request.num_nodes > self.total_nodes:
                raise StructuralRejection(
                    f"job {request.key!r} requests {request.num_nodes} nodes, "
                    f"but platform {self.name!r} has {self.total_nodes}"
                )
            if request.submit_s < current_time:
                raise ClockViolation(
                    f"job {request.key!r} submits at {request.submit_s}, "
                    f"before platform time {current_time}"
                )
            if (previous_submit is not None
                    and request.submit_s < previous_submit):
                raise ClockViolation(
                    f"job {request.key!r} submits at {request.submit_s}, "
                    f"before the previous submission at {previous_submit}"
                )
            previous_submit = request.submit_s

        if not requests:
            return []

        try:
            append_requests = [
                self._dr_evt.JobAppendRequest(
                    request.submit_s,
                    request.num_nodes,
                    request.q_id,
                    request.limit_s,
                )
                for request in requests
            ]
            handles = [
                int(handle)
                for handle in self._simulation.append_jobs(append_requests)
            ]
        except _DR_EVT_ERRORS as error:
            raise InfrastructureFailure(
                f"platform {self.name!r} rejected an append batch: {error}"
            ) from error
        if len(handles) != len(requests):
            raise InfrastructureFailure(
                f"platform {self.name!r} returned {len(handles)} handles "
                f"for {len(requests)} requests"
            )

        for handle, request in zip(handles, requests):
            self._ledger[handle] = request.key
            self._handles.append(handle)
        self._last_submit_s = previous_submit
        return handles
```

```python
    def advance_to(self, time_s: int) -> None:
        """Advance through events at time_s after enforcing a monotone clock."""
        self._ensure_open()
        if not _is_integer(time_s):
            raise ClockViolation("advance time must be an integer number of seconds")
        current_time = self.now()
        if time_s < current_time:
            raise ClockViolation(
                f"cannot advance platform {self.name!r} from "
                f"{current_time} back to {time_s}"
            )
        try:
            self._simulation.advance_to(time_s)
        except _DR_EVT_ERRORS as error:
            raise InfrastructureFailure(
                f"cannot advance platform {self.name!r}: {error}"
            ) from error
```

```python
    def snapshot(self) -> PlatformSnapshot:
        """Return capacity, queue, and utilization state."""
        self._ensure_open()
        try:
            return PlatformSnapshot(
                name=self.name,
                time_s=self.now(),
                total_nodes=self.total_nodes,
                free_nodes=int(self._simulation.get_available_nodes()),
                in_use_nodes=int(self._simulation.get_nodes_in_use()),
                waiting_jobs=int(self._simulation.get_active_job_count()),
                current_utilization=float(
                    self._simulation.get_current_utilization()
                ),
            )
        except _DR_EVT_ERRORS as error:
            raise InfrastructureFailure(
                f"cannot snapshot platform {self.name!r}: {error}"
            ) from error
```

```python
    def timings(self, handles: Sequence[int]) -> list[JobTiming]:
        """Return adapter timing values, preserving requested handle order."""
        self._ensure_open()
        requested = list(handles)
        for handle in requested:
            if handle not in self._ledger:
                raise KeyError(
                    f"platform {self.name!r} has no job handle {handle}"
                )
        try:
            raw_timings = self._simulation.get_job_timings(requested)
        except _DR_EVT_ERRORS as error:
            raise InfrastructureFailure(
                f"cannot read timings from platform {self.name!r}: {error}"
            ) from error

        return [
            JobTiming(
                handle=int(timing.job_idx),
                key=self._ledger[int(timing.job_idx)],
                submit_s=float(timing.submit_time),
                begin_s=float(timing.begin_time),
                end_s=float(timing.end_time),
                limit_s=int(timing.limit_time),
                actual_run_s=float(timing.actual_run_time),
                num_nodes=int(timing.num_nodes),
                scheduled=bool(timing.scheduled),
            )
            for timing in raw_timings
        ]
```

```python
    def finish(self) -> PlatformReport:
        """Drain all work, write both traces, and return final records."""
        if self._report is not None:
            return self._report

        self.advance_to(max(self.now(), _DRAIN_TIME_S))
        timings = tuple(self.timings(self._handles))
        try:
            self._simulation.write_simulated_trace()
            self._simulation.write_resource_trace(
                str(self._resource_trace_path)
            )
            raw_statistics = self._simulation.get_statistics()
        except _DR_EVT_ERRORS as error:
            raise InfrastructureFailure(
                f"cannot finish platform {self.name!r}: {error}"
            ) from error

        statistics = {
            field: float(getattr(raw_statistics, field))
            for field in _STATISTIC_FIELDS
        }
        self._report = PlatformReport(
            name=self.name,
            timings=timings,
            statistics=statistics,
            simulated_trace_path=str(self._simulated_trace_path),
            resource_trace_path=str(self._resource_trace_path),
        )
        return self._report
```

`GrpcPlatform(name, total_nodes, address, work_dir_on_server, session_name)` is the
same contract, method for method, over the session stream. `SessionClient` is
`ServerSession` with two additions: an `ErrorResponse` becomes one of the four error
types, chosen from the request kind and the message, and closing also removes the
generated-stub directory. `ServerProcess(binary, work_dir, address)` starts
`dr_evt_server` on a free port, waits for the channel, writes the server's output to
`server.log`, and stops it on exit.

In [13]:
show(grpc_adapter.SessionClient.call)
show(grpc_adapter.GrpcPlatform.snapshot)
show(grpc_adapter.GrpcPlatform.timings)
show(grpc_adapter._mapped_server_error)
show(server.ServerProcess.start)

```python
    def call(self, request: Any) -> Any:
        """Send one request and return its correlated server response."""
        if self._closed:
            raise InfrastructureFailure(
                f"{self.address}: session client is closed"
            )
        request.request_id = self._request_id
        self._request_id += 1
        self._outgoing.put(request)
        try:
            response = next(self._responses)
        except StopIteration as error:
            raise InfrastructureFailure(
                f"{self.address}: server closed the stream"
            ) from error
        except self._grpc.RpcError as error:
            raise InfrastructureFailure(
                f"{self.address}: gRPC request failed: {error.details()}"
            ) from error
        if response.request_id != request.request_id:
            raise InfrastructureFailure(
                f"{self.address}: response id {response.request_id} does not "
                f"match request id {request.request_id}"
            )
        if response.HasField("error"):
            request_kind = request.WhichOneof("request") or ""
            raise _mapped_server_error(
                request_kind,
                self.address,
                response.error.message,
            )
        return response
```

```python
    def snapshot(self) -> PlatformSnapshot:
        """Return capacity, queue, and utilization state from the server."""
        statistics = self._client.call(self._messages.ClientMessage(
            get_statistics=self._messages.GetStatisticsRequest()
        )).get_statistics
        current_utilization = self._client.call(self._messages.ClientMessage(
            get_current_utilization=(
                self._messages.GetCurrentUtilizationRequest()
            )
        )).get_current_utilization.utilization
        return PlatformSnapshot(
            name=self.name,
            time_s=int(statistics.current_time),
            total_nodes=self.total_nodes,
            free_nodes=int(statistics.nodes_available),
            in_use_nodes=int(statistics.nodes_in_use),
            waiting_jobs=int(statistics.jobs_waiting),
            current_utilization=float(current_utilization),
        )
```

```python
    def timings(self, handles: Sequence[int]) -> list[JobTiming]:
        """Return timing values for handles in their requested order."""
        requested = list(handles)
        response = self._client.call(self._messages.ClientMessage(
            get_job_timings=self._messages.GetJobTimingsRequest(
                job_idx=requested
            )
        ))
        return [
            JobTiming(
                handle=int(timing.job_idx),
                key=self._ledger[int(timing.job_idx)],
                submit_s=float(timing.submit_time),
                begin_s=float(timing.begin_time),
                end_s=float(timing.end_time),
                limit_s=int(timing.limit_time),
                actual_run_s=float(timing.actual_run_time),
                num_nodes=int(timing.num_nodes),
                scheduled=bool(timing.scheduled),
            )
            for timing in response.get_job_timings.timings
        ]
```

```python
def _mapped_server_error(
    request_kind: str,
    address: str,
    server_message: str,
) -> Exception:
    message = f"{address}: {server_message}"
    if request_kind == "get_job_timings" and "get_job_timing()" in message:
        return KeyError(message)
    if "submit_time" in server_message and "current" in server_message:
        return ClockViolation(message)
    if (
        server_message.startswith("Unknown ")
        or server_message.startswith("session_name must")
    ):
        return ConfigurationError(message)
    return InfrastructureFailure(message)
```

```python
    def start(self) -> "ServerProcess":
        """Start the process and wait until its gRPC channel is ready."""
        if self._process is not None:
            raise InfrastructureFailure("DR_EVT server process is already started")
        try:
            import grpc
        except ImportError as error:
            raise InfrastructureFailure(
                "Python gRPC dependencies are missing; install dr_evt_market[grpc]"
            ) from error

        try:
            self._log_file = self._log_path.open("w", encoding="utf-8")
            self._process = subprocess.Popen(
                [str(self.binary), self.address],
                cwd=self.work_dir,
                stdout=self._log_file,
                stderr=self._log_file,
                text=True,
            )
        except OSError as error:
            if self._log_file is not None:
                self._log_file.close()
                self._log_file = None
            raise InfrastructureFailure(
                f"cannot start DR_EVT server {self.binary}: {error}"
            ) from error

        channel = grpc.insecure_channel(self.address)
        try:
            grpc.channel_ready_future(channel).result(timeout=15)
        except grpc.FutureTimeoutError as error:
            self.stop()
            detail = self._log_tail()
            raise InfrastructureFailure(
                f"DR_EVT server did not become ready at {self.address}; "
                f"server.log tail: {detail!r}"
            ) from error
        finally:
            channel.close()
        return self
```

## 4. Decision 4: what is sold, and what a bid is

The whole market is this loop: there is a stream of jobs; at each window the market
takes the queue (for now the whole queue, a prefix rule can come later), runs the
auction on it, and hands each winner to the scheduler of the cluster it won. Everything
in this section exists to make that one sentence precise.

The auction sells the capacity that is free at the window, nothing more. It does not
sell a position in a queue and it does not promise a start time in the future. A winner
is submitted to the cluster it won, into that cluster's own queue, and dr_evt's
scheduler takes it from there. A job that does not win stays in the market's queue and
is offered again at the next window.

In this setup the market is the only thing that submits to a cluster, and it submits
only what fits in the free nodes it has just read. So a cluster's queue is empty when a
winner arrives, every job it receives fits, EASY starts them all at once, and nothing is
left waiting. That is why a winner starts at the window time here. It is a property of
the setup, not of dr_evt: if a cluster also received jobs from elsewhere, local users
or a background trace, its queue could hold a job blocked ahead of ours, EASY would let
the winner run only if it finished before that head's reservation, and the winner could
wait even though it fits. `waiting_jobs` in the snapshot and the start time in the
receipt are how the market would notice.

Two things follow from selling only free capacity. The mechanism needs no prediction of
when a cluster will free up; it reads free nodes and decides. And the receipt read
afterwards should say "started at the window", which is the one check the loop makes.

### Feasibility is public

A cluster has hardware, an NVIDIA or AMD GPU, a fast interconnect, and a job may need
some of it. That is not a preference, it is a fact, so it does not belong in a bid. A
`Platform` carries a set of `hardware` tags and a `Leg` carries the tags it `requires`;
a leg fits a cluster when its tags are a subset of the cluster's and its nodes fit. A
placement that does not fit is never listed, so nobody bids on it and no mechanism can
choose it.

### The value model follows the clusters' pricing

A cluster posts a price per node hour, so the cost of a leg on a cluster is public:
price times nodes times the requested limit in hours. A job's base cost is what it
would pay on the cheapest placement it can use. Its bid is one number, a multiplier on
that base: `4` means "this run is worth up to four times my base cost to me". The value
is the same on every placement the job can use, so with one bid a job prefers cheaper
clusters, since net value, value minus cost, is larger there. A job that values
clusters differently gives one multiplier per cluster instead, and then the value of a
placement is the sum over its legs of the leg's cost on its cluster times that
cluster's multiplier; clusters without a multiplier are not used. A placement whose
value does not cover its cost is never offered. Whatever the mechanism, a winner pays
its placement's cost plus a premium, and the premium is what the mechanism sets.

A composite job has several legs that must start together, possibly on different
clusters. It has one bid, its value on a placement follows the same rule, and it is
placed as a whole or not at all.

### The eight records

| record | field | meaning |
|---|---|---|
| `Platform` | `name`, `total_nodes` | identity and size, the same as the session's |
| | `price_per_node_hour` | the posted price; `cost(num_nodes, limit_s)` is what a leg costs here |
| | `hardware` | tags a leg may require |
| | `address` | a gRPC address, or `None` for in process |
| `Leg` | `leg_id` | the leg's name within its job, `"0"` for a one-leg job |
| | `num_nodes`, `limit_s` | node demand and time limit, what dr_evt is told |
| | `requires` | hardware tags; `fits(platform)` checks tags and nodes |
| `Job` | `job_id`, `submit_s` | identity and arrival time |
| | `legs` | the legs in order |
| | `bid` | one multiplier on the base cost, or a mapping from cluster to multiplier; `multiplier(platform)` reads it |
| `Placement` | `platforms` | one cluster per leg, in leg order; `id` joins them with `+` |
| | `cost_credits` | the public cost, the sum of the legs' costs |
| | `value_credits` | the job's value on it, from the bid; `net_credits` is the difference |
| `Window` | `time_s`, `index` | when, and which window |
| | `platforms`, `free_nodes` | the clusters and the supply the snapshots reported |
| | `jobs` | the queue, in arrival order |
| | `candidates` | per job, the placements that fit in the free nodes now and cover their cost |
| `Decision` | `job_id`, `placement`, `charge_credits` | one job, one of its candidates, what it pays |
| `Rejection` | `job_id`, `reason`, `time_s` | why a job or a decision was not placed |
| `Mechanism` | `name`, `decide(window)` | the abstract base every mechanism extends |

In [14]:
show(mechanism_base.Platform)
show(mechanism_base.Leg)
show(mechanism_base.Job)

```python
@dataclass(frozen=True)
class Platform:
    """One cluster as the market sees it: size, price and hardware."""

    name: str
    total_nodes: int
    price_per_node_hour: float
    hardware: frozenset[str] = frozenset()
    address: str | None = None

    def __post_init__(self) -> None:
        _name(self.name, "platform name")
        _positive_int(self.total_nodes, "total_nodes")
        _money(self.price_per_node_hour, "price_per_node_hour")
        object.__setattr__(self, "hardware", frozenset(self.hardware))

    def cost(self, num_nodes: int, limit_s: int) -> float:
        """Return the posted cost in credits of nodes held for a time limit."""
        return self.price_per_node_hour * num_nodes * limit_s / 3600.0
```

```python
@dataclass(frozen=True)
class Leg:
    """One part of a job: nodes, time limit and the hardware it needs."""

    leg_id: str
    num_nodes: int
    limit_s: int
    requires: frozenset[str] = frozenset()

    def __post_init__(self) -> None:
        _name(self.leg_id, "leg_id")
        _positive_int(self.num_nodes, "num_nodes")
        _positive_int(self.limit_s, "limit_s")
        object.__setattr__(self, "requires", frozenset(self.requires))

    def fits(self, platform: Platform) -> bool:
        """Return whether the platform has the hardware and the nodes."""
        return (
            self.requires <= platform.hardware
            and self.num_nodes <= platform.total_nodes
        )
```

```python
@dataclass(frozen=True)
class Job:
    """One job of the stream: its legs and its bid.

    ``bid`` is a multiplier on the job's base cost, the cost of its cheapest
    usable placement, so ``1.5`` means the job would pay up to one and a half
    times its base cost wherever it runs. A mapping from platform name to
    multiplier values each platform separately: the value of a placement is
    then the sum over legs of that leg's cost on its platform times the
    platform's multiplier, and platforms without a multiplier are not used.
    """

    job_id: str
    submit_s: int
    legs: tuple[Leg, ...]
    bid: float | Mapping[str, float]

    def __post_init__(self) -> None:
        _name(self.job_id, "job_id")
        if not isinstance(self.submit_s, int) or isinstance(self.submit_s, bool):
            raise ValueError("submit_s must be an integer")
        if self.submit_s < 0:
            raise ValueError("submit_s must be non-negative")
        legs = tuple(self.legs)
        if not legs:
            raise ValueError("a job needs at least one leg")
        if len({leg.leg_id for leg in legs}) != len(legs):
            raise ValueError("leg ids must be unique within a job")
        object.__setattr__(self, "legs", legs)
        if isinstance(self.bid, Mapping):
            bid = {
                _name(name, "platform name"): _money(value, "bid")
                for name, value in self.bid.items()
            }
            if not bid:
                raise ValueError("a per-platform bid needs at least one platform")
            object.__setattr__(self, "bid", dict(sorted(bid.items())))
        else:
            object.__setattr__(self, "bid", _money(self.bid, "bid"))

    def multiplier(self, platform: str) -> float | None:
        """Return the multiplier that applies on a platform, or None."""
        if isinstance(self.bid, Mapping):
            return self.bid.get(platform)
        return self.bid
```

In [15]:
show(mechanism_base.Placement)
show(mechanism_base.Window)
show(mechanism_base.Decision)
show(mechanism_base.Rejection)
show(mechanism_base.Mechanism)

```python
@dataclass(frozen=True)
class Placement:
    """One way to run a job: a platform per leg, its cost and its value."""

    platforms: tuple[str, ...]
    cost_credits: float
    value_credits: float

    def __post_init__(self) -> None:
        platforms = tuple(_name(name, "platform name") for name in self.platforms)
        if not platforms:
            raise ValueError("a placement needs at least one platform")
        object.__setattr__(self, "platforms", platforms)
        object.__setattr__(self, "cost_credits", _money(self.cost_credits, "cost"))
        object.__setattr__(self, "value_credits", _money(self.value_credits, "value"))

    @property
    def id(self) -> str:
        """Return the platforms joined with ``+`` in leg order."""
        return "+".join(self.platforms)

    @property
    def net_credits(self) -> float:
        """Return value minus cost."""
        return self.value_credits - self.cost_credits
```

```python
@dataclass(frozen=True)
class Window:
    """One clearing window: the platforms, their free nodes, the queue, the
    candidates of every queued job."""

    time_s: int
    index: int
    platforms: Mapping[str, Platform]
    free_nodes: Mapping[str, int]
    jobs: tuple[Job, ...]
    candidates: Mapping[str, tuple[Placement, ...]]

    def __post_init__(self) -> None:
        jobs = tuple(self.jobs)
        if len({job.job_id for job in jobs}) != len(jobs):
            raise ValueError("job ids must be unique within a window")
        object.__setattr__(self, "jobs", jobs)
        object.__setattr__(self, "platforms", dict(sorted(self.platforms.items())))
        if set(self.free_nodes) != set(self.platforms):
            raise ValueError("free_nodes must name exactly the window's platforms")
        object.__setattr__(self, "free_nodes", dict(sorted(self.free_nodes.items())))
        candidates = {
            job.job_id: tuple(self.candidates.get(job.job_id, ())) for job in jobs
        }
        object.__setattr__(self, "candidates", candidates)

    def job(self, job_id: str) -> Job:
        """Return the job with this id or raise KeyError."""
        for job in self.jobs:
            if job.job_id == job_id:
                return job
        raise KeyError(job_id)
```

```python
@dataclass(frozen=True)
class Decision:
    """Give one job one of its candidate placements at a charge."""

    job_id: str
    placement: Placement
    charge_credits: float
```

```python
@dataclass(frozen=True)
class Rejection:
    """Record why a job or a decision was not placed."""

    job_id: str
    reason: str
    time_s: int
```

```python
class Mechanism(ABC):
    """Decide who runs where and what they pay, one window at a time."""

    name: str

    @abstractmethod
    def decide(self, window: Window) -> list[Decision]:
        """Return at most one decision per job, each on one of its candidates."""
```

### The functions between them

| function | meaning |
|---|---|
| `placements(job, platforms, free_nodes)` | every placement the job could take: each leg on a cluster with its hardware, its nodes and, when free nodes are given, room now; legs sharing a cluster fit together; clusters in name order so the list is deterministic; with free nodes given, placements whose value is below cost are left out |
| `base_cost(job, platforms)` | the cost of the cheapest placement the job can use |
| `build_window(time_s, index, queue, platforms, free_nodes)` | the queue and the supply into a `Window`, with the candidates of every job |
| `demand(job, placement)` | nodes per cluster, legs on the same cluster summed |
| `validate_decisions(window, decisions)` | keep the decisions the window can honour, in decision order, and name every refusal |
| `submit(decisions, window, sessions, time_s)` | one batch per cluster, one handle per leg |

Because a mechanism is a plug, its output is never trusted. `validate_decisions`
checks it with a running per-cluster demand; a refused decision consumes no capacity,
so a mechanism that overbooks degrades in a defined way instead of crashing the run.

| reason | meaning |
|---|---|
| `duplicate_job` | a second decision for a job in the same window |
| `unknown_job`, `unknown_placement` | not in the window |
| `charge_above_value`, `charge_below_cost` | outside the allowed interval |
| `over_capacity` | cumulative demand on a cluster exceeds its free nodes |

In [16]:
show(clearing.placements)
show(clearing.build_window)
show(mechanism_base.validate_decisions)
show(clearing.submit)

```python
def placements(
    job: Job,
    platforms: Mapping[str, Platform],
    free_nodes: Mapping[str, int] | None = None,
) -> tuple[Placement, ...]:
    """Return every placement the job could take, with cost and value.

    A leg may go to a platform that has its hardware, enough nodes overall and,
    when ``free_nodes`` is given, enough free nodes now; a job with a
    per-platform bid may only use the platforms it bid on. Legs sharing a
    platform must fit together. Platforms are tried in name order, so the
    result is deterministic. With ``free_nodes`` given, placements whose value
    does not cover their cost are left out, since no mechanism may take them.
    """
    names = sorted(platforms)
    choices = []
    for leg in job.legs:
        usable = [
            name
            for name in names
            if leg.fits(platforms[name])
            and job.multiplier(name) is not None
            and (free_nodes is None or leg.num_nodes <= free_nodes.get(name, 0))
        ]
        if not usable:
            return ()
        choices.append(usable)

    single = not isinstance(job.bid, Mapping)
    base = base_cost(job, platforms) if single else None
    found = []
    for assignment in product(*choices):
        nodes: dict[str, int] = {}
        cost = 0.0
        value = 0.0
        for leg, name in zip(job.legs, assignment):
            nodes[name] = nodes.get(name, 0) + leg.num_nodes
            leg_cost = platforms[name].cost(leg.num_nodes, leg.limit_s)
            cost += leg_cost
            if not single:
                value += job.multiplier(name) * leg_cost
        if any(
            count > platforms[name].total_nodes
            or (free_nodes is not None and count > free_nodes.get(name, 0))
            for name, count in nodes.items()
        ):
            continue
        if single:
            value = job.bid * base
        if free_nodes is not None and value < cost:
            continue
        found.append(Placement(tuple(assignment), cost, value))
    return tuple(found)
```

```python
def build_window(
    time_s: int,
    index: int,
    queue: Sequence[Job],
    platforms: Mapping[str, Platform],
    free_nodes: Mapping[str, int],
) -> Window:
    """Offer every queued job its placements in the free nodes of this window."""
    return Window(
        time_s,
        index,
        dict(platforms),
        dict(free_nodes),
        tuple(queue),
        {job.job_id: placements(job, platforms, free_nodes) for job in queue},
    )
```

```python
def validate_decisions(
    window: Window,
    decisions: Sequence[Decision],
) -> tuple[list[Decision], list[Rejection]]:
    """Keep the decisions the window can honour, in order, and say why not.

    A job's first decision is the only one considered. A decision is refused
    for ``duplicate_job``, ``unknown_job``, ``unknown_placement``,
    ``charge_above_value``, ``charge_below_cost`` or ``over_capacity``; a
    refused decision consumes no capacity.
    """
    accepted: list[Decision] = []
    rejected: list[Rejection] = []
    seen: set[str] = set()
    used: dict[str, int] = {name: 0 for name in window.free_nodes}

    for decision in decisions:
        reason = None
        job = None
        if decision.job_id in seen:
            reason = "duplicate_job"
        else:
            seen.add(decision.job_id)
            try:
                job = window.job(decision.job_id)
            except KeyError:
                reason = "unknown_job"
        if reason is None and decision.placement not in window.candidates[job.job_id]:
            reason = "unknown_placement"
        if reason is None:
            charge = float(decision.charge_credits)
            placement = decision.placement
            if not isfinite(charge) or charge > placement.value_credits + 1e-9:
                reason = "charge_above_value"
            elif charge < placement.cost_credits - 1e-9:
                reason = "charge_below_cost"
            else:
                needed = demand(job, placement)
                if any(
                    used.get(name, 0) + nodes > window.free_nodes.get(name, 0)
                    for name, nodes in needed.items()
                ):
                    reason = "over_capacity"
        if reason is not None:
            rejected.append(Rejection(decision.job_id, reason, window.time_s))
            continue
        accepted.append(decision)
        for name, nodes in demand(job, decision.placement).items():
            used[name] = used.get(name, 0) + nodes
    return accepted, rejected
```

```python
def submit(
    decisions: Sequence[Decision],
    window: Window,
    sessions: Mapping[str, PlatformSession],
    time_s: int,
) -> dict[tuple[str, str], int]:
    """Send every accepted leg to its platform, one batch per platform.

    Returns the platform handle of every leg, keyed by job id and leg id.
    """
    names = sorted(sessions)
    batches: dict[str, list[SubmitRequest]] = {name: [] for name in names}
    keys: dict[str, list[tuple[str, str]]] = {name: [] for name in names}
    for decision in decisions:
        job = window.job(decision.job_id)
        for leg, name in zip(job.legs, decision.placement.platforms):
            if name not in batches:
                raise ValueError(f"unknown platform {name!r}")
            batches[name].append(
                SubmitRequest(
                    key=f"{job.job_id}/{leg.leg_id}",
                    submit_s=time_s,
                    num_nodes=leg.num_nodes,
                    limit_s=leg.limit_s,
                )
            )
            keys[name].append((job.job_id, leg.leg_id))
    handles: dict[tuple[str, str], int] = {}
    for name in names:
        if not batches[name]:
            continue
        returned = sessions[name].submit(batches[name])
        if len(returned) != len(keys[name]):
            raise InfrastructureFailure(
                f"platform {name!r} returned {len(returned)} handles "
                f"for {len(keys[name])} legs"
            )
        handles.update(zip(keys[name], returned))
    return handles
```

## 5. Decision 5: the mechanism as a plug, VCG first

VCG is the first mechanism because it is the reference: it maximizes the total net
value placed and it is truthful, so bidding one's real multiplier is the best strategy.
It solves the window exactly as a MILP, one binary per (job, candidate), one placement
per job, per-cluster capacity, to zero gap. The premium it charges a winner is the
Clarke pivot: solve the window again without the winner, and charge what the others
would have gained. A winner that displaces nobody pays cost only.

In [17]:
show(vcg.Vcg.decide)

```python
    def decide(self, window: Window) -> list[Decision]:
        """Return the welfare-maximizing decisions with pivot charges."""
        chosen, welfare = _solve(window, self.time_limit_s)
        decisions = []
        for job in window.jobs:
            placement = chosen.get(job.job_id)
            if placement is None:
                continue
            _, without = _solve(window, self.time_limit_s, excluded=job.job_id)
            pivot = max(without - (welfare - placement.net_credits), 0.0)
            charge = min(placement.value_credits, placement.cost_credits + pivot)
            decisions.append(Decision(job.job_id, placement, charge))
        return decisions
```

## 6. Decision 6: windows, the loop, and the outputs

Clearing at fixed boundaries is the simplest cadence that makes the market well defined:
jobs that arrive between boundaries wait for the next one, so every decision sees the
same kind of state. `Controller(sessions, platforms, mechanism, jobs, window_s, seed)`
runs that loop. At each boundary it advances every cluster, admits arrivals, takes one
snapshot per cluster, builds the window, asks the mechanism, validates, submits,
advances again to the same time, and reads the receipts. Intake rejects jobs that fit
nowhere (`oversize`) or whose value is below cost on every placement (`unaffordable`).
The run ends when arrivals and the queue are exhausted, every cluster is finished, and
the receipts are joined to the decisions.

Every output is written deterministically and hashed, so two runs can be compared by
one line.

| record | field | meaning |
|---|---|---|
| `RoutedLeg` | `job_id`, `leg_id`, `platform` | which leg went where |
| | `window_index`, `window_time_s` | the window that placed it |
| | `handle` | dr_evt's job index on that cluster |
| | `cost_credits`, `value_credits`, `premium_credits`, `charge_credits` | the placement's cost and value, the premium the mechanism set, and cost plus premium; a composite repeats them on each leg |
| | `submit_s`, `begin_s`, `end_s` | arrival, and the start and end from the receipt |
| `WindowRecord` | `index`, `time_s` | the window |
| | `queued`, `placed` | job ids waiting and job ids placed |
| | `rejected` | decisions the validator refused, with reasons |
| | `welfare`, `revenue` | net value placed and charges collected |
| | `free_nodes` | the supply the window saw |
| `RunReport` | `routed`, `windows`, `rejected`, `reports`, `configuration` | all of the above, every `Rejection`, each cluster's `PlatformReport` and the run settings |

A `RoutingError` is raised if a receipt says a winner did not begin at its window, the
one assumption of the setup that the loop checks.

In [18]:
show(controller.RoutedLeg)
show(controller.WindowRecord)
show(controller.RunReport)

```python
@dataclass(frozen=True)
class RoutedLeg:
    """One leg that ran: where it went, what the job paid, what happened."""

    job_id: str
    leg_id: str
    platform: str
    window_index: int
    window_time_s: int
    handle: int
    cost_credits: float
    value_credits: float
    premium_credits: float
    charge_credits: float
    submit_s: int
    begin_s: int
    end_s: int
```

```python
@dataclass(frozen=True)
class WindowRecord:
    """What one window saw and did."""

    index: int
    time_s: int
    queued: tuple[str, ...]
    placed: tuple[str, ...]
    rejected: tuple[Rejection, ...]
    welfare: float
    revenue: float
    free_nodes: dict[str, int]
```

```python
@dataclass
class RunReport:
    """Everything a run produced."""

    routed: list[RoutedLeg]
    windows: list[WindowRecord]
    rejected: list[Rejection]
    reports: dict[str, PlatformReport]
    configuration: dict[str, object] = field(default_factory=dict)
```

In [19]:
show(controller.Controller.run)

```python
    def run(self) -> RunReport:
        """Run every window through the drain and return the records."""
        if self._report is not None:
            return self._report
        arrivals = []
        rejected = []
        for job in self.jobs:
            reason = self._intake(job)
            if reason is None:
                arrivals.append(job)
            else:
                rejected.append(Rejection(job.job_id, reason, job.submit_s))

        queue: list[Job] = []
        pending: list[tuple] = []
        windows: list[WindowRecord] = []
        next_arrival = 0
        index = 0
        time_s = 0
        while next_arrival < len(arrivals) or queue:
            for session in self.sessions.values():
                session.advance_to(time_s)
            while (
                next_arrival < len(arrivals)
                and arrivals[next_arrival].submit_s <= time_s
            ):
                queue.append(arrivals[next_arrival])
                next_arrival += 1
            snapshots = {
                name: session.snapshot() for name, session in self.sessions.items()
            }
            window = build_window(
                time_s,
                index,
                queue,
                self.platforms,
                {name: snapshot.free_nodes for name, snapshot in snapshots.items()},
            )
            if self.log_dir is not None:
                _write_window(self.log_dir, window)
            accepted, refused = validate_decisions(
                window, self.mechanism.decide(window)
            )
            handles = submit(accepted, window, self.sessions, time_s)
            for session in self.sessions.values():
                session.advance_to(time_s)
            timings = self._timings(accepted, window, handles)
            for decision in accepted:
                job = window.job(decision.job_id)
                for leg, name in zip(job.legs, decision.placement.platforms):
                    timing = timings[(job.job_id, leg.leg_id)]
                    if not timing.scheduled or timing.begin_s != float(time_s):
                        raise RoutingError(
                            f"{job.job_id}/{leg.leg_id} began at {timing.begin_s}, "
                            f"not at its window {time_s}"
                        )
                    pending.append(
                        (
                            decision,
                            job,
                            leg,
                            name,
                            index,
                            time_s,
                            handles[(job.job_id, leg.leg_id)],
                        )
                    )
            placed = tuple(decision.job_id for decision in accepted)
            windows.append(
                WindowRecord(
                    index,
                    time_s,
                    tuple(job.job_id for job in queue),
                    placed,
                    tuple(refused),
                    sum(decision.placement.net_credits for decision in accepted),
                    sum(decision.charge_credits for decision in accepted),
                    {name: snapshot.free_nodes for name, snapshot in snapshots.items()},
                )
            )
            queue = [job for job in queue if job.job_id not in set(placed)]
            if (
                not accepted
                and queue
                and all(
                    snapshot.free_nodes == snapshot.total_nodes
                    for snapshot in snapshots.values()
                )
            ):
                rejected.extend(
                    Rejection(job.job_id, "unplaceable", time_s) for job in queue
                )
                queue.clear()
                break
            index += 1
            time_s += self.window_s

        reports = {name: session.finish() for name, session in self.sessions.items()}
        final = {
            (name, timing.handle): timing
            for name, report in reports.items()
            for timing in report.timings
        }
        routed = []
        for decision, job, leg, name, window_index, window_time, handle in pending:
            timing = final[(name, handle)]
            routed.append(
                RoutedLeg(
                    job.job_id,
                    leg.leg_id,
                    name,
                    window_index,
                    window_time,
                    handle,
                    decision.placement.cost_credits,
                    decision.placement.value_credits,
                    decision.charge_credits - decision.placement.cost_credits,
                    decision.charge_credits,
                    job.submit_s,
                    _whole_seconds(timing.begin_s),
                    _whole_seconds(timing.end_s),
                )
            )
        self._report = RunReport(
            routed,
            windows,
            rejected,
            reports,
            {
                "mechanism": self.mechanism.name,
                "seed": self.seed,
                "window_s": self.window_s,
                "platforms": {
                    name: {
                        "total_nodes": platform.total_nodes,
                        "price_per_node_hour": platform.price_per_node_hour,
                        "hardware": sorted(platform.hardware),
                    }
                    for name, platform in self.platforms.items()
                },
            },
        )
        return self._report
```

## 7. The run

### The input files

Both files mirror dr_evt's own. `platforms.csv` follows `sync_systems.csv`: a system
id, a node count, the posted price per node hour, the hardware tags, and a blank or
gRPC address. `jobs.csv` is a dr_evt trace, `job_submit_time`, `num_nodes`,
`time_limit`, with a `job_id`, a `leg_id` (one row per leg, as in dr_evt's composite
format), the hardware a leg `requires`, and the `bid`, the job's multiplier on its
base cost. Two jobs need a GPU, so they can only go to gamma; `c01` is composite, its
right leg needs a GPU; `s12` bids half its base cost and is rejected at intake.

In [20]:
display(pd.read_csv(EXAMPLES / "market_platforms.csv"))
pd.read_csv(EXAMPLES / "market_jobs.csv")

,system_id,total_nodes,price_per_node_hour,hardware,address
0,alpha,100,1.0,cpu,NaN
1,beta,60,2.0,cpu,NaN
2,gamma,40,3.0,cpu gpu,NaN


,job_id,job_submit_time,num_nodes,time_limit,leg_id,requires,bid
0,s01,0,70,120,0,NaN,4.0
1,s02,0,50,180,0,NaN,4.0
2,s03,0,40,90,0,gpu,4.0
3,s04,0,40,100,0,NaN,4.0
4,s05,30,20,80,0,NaN,3.0
5,s06,60,10,60,0,NaN,3.0
6,s07,120,60,100,0,NaN,4.0
7,s08,180,30,120,0,NaN,3.0
8,c01,240,30,100,left,NaN,3.0
9,c01,240,20,100,right,gpu,3.0


In [21]:
specs = m.read_platforms(EXAMPLES / "market_platforms.csv")
platforms = {p.name: p for p in specs}
jobs = m.read_jobs(EXAMPLES / "market_jobs.csv")
print(len(jobs), "jobs,", sum(len(j.legs) for j in jobs), "legs;", "composite:", [j.job_id for j in jobs if len(j.legs) > 1])
for job in jobs[:4]:
    base = clearing.base_cost(job, platforms)
    print(f"{job.job_id}: bid {job.bid:g} x base cost {base:.2f} = value {job.bid * base:.2f} on",
          [p.id for p in clearing.placements(job, platforms)])

13 jobs, 14 legs; composite: ['c01']
s01: bid 4 x base cost 2.33 = value 9.33 on ['alpha']
s02: bid 4 x base cost 2.50 = value 10.00 on ['alpha', 'beta']
s03: bid 4 x base cost 3.00 = value 12.00 on ['gamma']
s04: bid 4 x base cost 1.11 = value 4.44 on ['alpha', 'beta', 'gamma']


### One window by hand

At time 0 four jobs have arrived, wanting 200 nodes between them, exactly the
federation's capacity, but no assignment fits all four. `build_window` lists every
candidate with its cost, its value, and the net value the mechanism maximizes. `s01`
needs 70 nodes and only alpha has them; `s03` needs a GPU and only gamma has one;
`s04` is small enough to go anywhere.

In [22]:
WORK = Path(tempfile.mkdtemp(dir=OUT, prefix="market07_"))
sessions = {p.name: m.InProcessPlatform(p.name, p.total_nodes, WORK / "hand" / p.name) for p in specs}
for s in sessions.values():
    s.advance_to(0)
queue = [j for j in jobs if j.submit_s <= 0]
window = clearing.build_window(0, 0, queue, platforms, {n: s.snapshot().free_nodes for n, s in sessions.items()})
rows = []
for job in window.jobs:
    for c in window.candidates[job.job_id]:
        rows.append({"job": job.job_id, "placement": c.id, "nodes": mechanism_base.demand(job, c),
                     "cost": round(c.cost_credits, 2), "value": round(c.value_credits, 2), "net": round(c.net_credits, 2)})
print("free nodes:", dict(window.free_nodes))
pd.DataFrame(rows)

free nodes: {'alpha': 100, 'beta': 60, 'gamma': 40}


,job,placement,nodes,cost,value,net
0,s01,alpha,{'alpha': 70},2.33,9.33,7.00
1,s02,alpha,{'alpha': 50},2.50,10.00,7.50
2,s02,beta,{'beta': 50},5.00,10.00,5.00
3,s03,gamma,{'gamma': 40},3.00,12.00,9.00
4,s04,alpha,{'alpha': 40},1.11,4.44,3.33
5,s04,beta,{'beta': 40},2.22,4.44,2.22
6,s04,gamma,{'gamma': 40},3.33,4.44,1.11


VCG places three of the four and defers `s04`. Alpha cannot hold `s01` and `s02`
together, so `s02` goes to beta at twice the cost, since that is worth more in total
than leaving `s01` out. Each winner pays its cluster's cost plus a premium, and the
premium is the net value `s04` would have had on that winner's cluster, because that is
exactly what the others lose by the winner being there.

In [23]:
decisions = vcg.Vcg().decide(window)
accepted, refused = mechanism_base.validate_decisions(window, decisions)
print("net value placed:", round(sum(d.placement.net_credits for d in accepted), 2), "| refused:", refused)
pd.DataFrame([{"job": d.job_id, "placement": d.placement.id, "cost": round(d.placement.cost_credits, 2),
               "value": round(d.placement.value_credits, 2), "premium": round(d.charge_credits - d.placement.cost_credits, 2),
               "charge": round(d.charge_credits, 2)} for d in accepted])

net value placed: 21.0 | refused: []


,job,placement,cost,value,premium,charge
0,s01,alpha,2.33,9.33,5.83,8.17
1,s02,beta,5.00,10.00,2.22,7.22
2,s03,gamma,3.00,12.00,1.11,4.11


The winners are submitted to the clusters they won, at the window time; advancing each
cluster to that same time lets its scheduler evaluate its queue. The receipts then show
what the clusters did.

In [24]:
handles = clearing.submit(accepted, window, sessions, 0)
for s in sessions.values():
    s.advance_to(0)
for decision in accepted:
    job = window.job(decision.job_id)
    for leg, name in zip(job.legs, decision.placement.platforms):
        t = sessions[name].timings([handles[(job.job_id, leg.leg_id)]])[0]
        print(f"{job.job_id}/{leg.leg_id} on {name}: begin {t.begin_s:.0f} end {t.end_s:.0f} (limit {t.limit_s})")
print("waiting after submission:", {n: s.snapshot().waiting_jobs for n, s in sessions.items()})
for s in sessions.values():
    s.finish()

s01/0 on alpha: begin 0 end 120 (limit 120)
s02/0 on beta: begin 0 end 180 (limit 180)
s03/0 on gamma: begin 0 end 90 (limit 90)
waiting after submission: {'alpha': 0, 'beta': 0, 'gamma': 0}


Every winner began at the window, and each cluster's queue is empty again because every
job it received fitted and started at once. `s04` is still in the market's queue and
will be offered again at the next window.

### The whole run

In [25]:
sessions = {p.name: m.InProcessPlatform(p.name, p.total_nodes, WORK / "run" / p.name) for p in specs}
report = m.Controller(sessions, platforms, vcg.Vcg(), jobs, window_s=60, seed=0).run()
paths = m.write_outputs(report, WORK / "run" / "out")
print("rejected:", [(r.job_id, r.reason, r.time_s) for r in report.rejected])
pd.DataFrame([{"window": w.index, "time_s": w.time_s, "queued": list(w.queued), "placed": list(w.placed),
               "net value": round(w.welfare, 1), "charges": round(w.revenue, 1), "free": w.free_nodes} for w in report.windows])

rejected: [('s12', 'unaffordable', 600)]


,window,time_s,queued,placed,net value,charges,free
0,0,0,"[s01, s02, s03, s04]","[s01, s02, s03]",21.0,19.5,"{'alpha': 100, 'beta': 60, 'gamma': 40}"
1,1,60,"[s04, s05, s06]","[s05, s06]",1.2,0.6,"{'alpha': 30, 'beta': 10, 'gamma': 0}"
2,2,120,"[s04, s07]","[s04, s07]",6.1,7.2,"{'alpha': 80, 'beta': 10, 'gamma': 40}"
3,3,180,[s08],[s08],2.0,1.0,"{'alpha': 40, 'beta': 60, 'gamma': 0}"
4,4,240,[c01],[c01],5.0,2.5,"{'alpha': 70, 'beta': 60, 'gamma': 40}"
5,5,300,[s09],[s09],3.3,1.7,"{'alpha': 70, 'beta': 60, 'gamma': 20}"
6,6,360,[s10],[s10],3.8,1.9,"{'alpha': 60, 'beta': 60, 'gamma': 40}"
7,7,420,[],[],0.0,0.0,"{'alpha': 60, 'beta': 60, 'gamma': 15}"
8,8,480,[s11],[s11],2.1,1.1,"{'alpha': 100, 'beta': 60, 'gamma': 40}"


In [26]:
routed = pd.read_csv(paths["routed"])
routed[["job_id", "leg_id", "platform", "window_time_s", "submit_s", "begin_s", "end_s",
        "cost_credits", "value_credits", "premium_credits", "charge_credits"]].round(2)

,job_id,leg_id,platform,window_time_s,submit_s,begin_s,end_s,cost_credits,value_credits,premium_credits,charge_credits
0,s01,0,alpha,0,0,0,120,2.33,9.33,5.83,8.17
1,s02,0,beta,0,0,0,180,5.00,10.00,2.22,7.22
2,s03,0,gamma,0,0,0,90,3.00,12.00,1.11,4.11
3,s05,0,alpha,60,30,60,140,0.44,1.33,0.00,0.44
4,s06,0,alpha,60,60,60,120,0.17,0.50,0.00,0.17
5,s04,0,gamma,120,0,120,220,3.33,4.44,0.00,3.33
6,s07,0,alpha,120,120,120,220,1.67,6.67,2.22,3.89
7,s08,0,alpha,180,180,180,300,1.00,3.00,0.00,1.00
8,c01,left,alpha,240,240,240,340,2.50,7.50,0.00,2.50
9,c01,right,gamma,240,240,240,340,2.50,7.50,0.00,2.50


`s04` waited two windows and went to gamma at 120, the only cluster with room for it
then; `s07` paid a premium at 120 because it took the alpha nodes `s04` would have used;
the composite `c01` was split across alpha and gamma with both legs starting together;
every other winner paid its cluster's cost only. The statistics below are dr_evt's own,
per cluster.

In [27]:
wait = routed.begin_s - routed.submit_s
print("legs:", len(routed), "| mean wait for a window:", round(wait.mean(), 1), "s | max:", wait.max(), "s")
pd.DataFrame({n: {k: round(v, 3) for k, v in r.statistics.items() if k in ("jobs_completed", "makespan", "avg_wait_time", "utilization")} for n, r in report.reports.items()}).T

legs: 13 | mean wait for a window: 11.5 s | max: 120 s


,jobs_completed,utilization,avg_wait_time,makespan
alpha,8.0,0.560,0.0,590.0
beta,1.0,0.833,0.0,180.0
gamma,4.0,0.658,0.0,450.0


### One cluster over gRPC

The controller does not know which transport a cluster uses. Here alpha is served by a
`dr_evt_server` that the package starts, and the run produces the same `routed.csv`.

In [28]:
with m.ServerProcess(None, WORK / "grpc" / "server") as srv:
    mixed = {}
    for p in specs:
        if p.name == "alpha":
            mixed[p.name] = m.GrpcPlatform(p.name, p.total_nodes, srv.address, WORK / "grpc" / "server", session_name="alpha")
        else:
            mixed[p.name] = m.InProcessPlatform(p.name, p.total_nodes, WORK / "grpc" / p.name)
    report_grpc = m.Controller(mixed, platforms, vcg.Vcg(), jobs, window_s=60, seed=0).run()
    paths_grpc = m.write_outputs(report_grpc, WORK / "grpc" / "out")
print("in process:", paths["routed_sha256"][:16], "| alpha over gRPC:", paths_grpc["routed_sha256"][:16])

in process: 2328e1fe136e0405 | alpha over gRPC: 2328e1fe136e0405


### The command line

`python -m dr_evt_market run` does the same from a shell and prints the counts and the
two hashes; `--log-windows` writes one window per file for training.

In [29]:
result = subprocess.run([sys.executable, "-m", "dr_evt_market", "run",
                         "--jobs", str(EXAMPLES / "market_jobs.csv"), "--platforms", str(EXAMPLES / "market_platforms.csv"),
                         "--out", str(WORK / "cli"), "--window", "60", "--log-windows"],
                        capture_output=True, text=True, env={**os.environ, "PYTHONPATH": os.pathsep.join(sys.path[:2])})
print(result.stdout.strip())
print("logged windows:", len(list((WORK / "cli").glob("window_*.json"))))

windows=9
routed=13
rejected=1
routed_sha256=2328e1fe136e040566d52fde52f00b6115f73b3e7109476418bf363bfb41e6ba
windows_sha256=0a0517348d69eeed2280e14c2a85178af553e7ac974602ca4cc41d6fe40b801d
logged windows: 9


## 8. RegretFormer, the learned mechanism

Because the mechanism is a plug, a learned one drops in without touching the rest.
`RegretFormer` sees the window as a (job, candidate) grid with one private channel, the
candidate's value, and five public ones; it outputs allocation probabilities and a
payment fraction; deployment rounds greedily under free nodes and charges at most the
value, so a winner still pays cost plus a premium. A job's report is its multipliers,
so a misreport is always a bid the job could have made. Its regret, how much a job
could gain by misreporting, is estimated by a fixed grid search over each multiplier
followed by gradient ascent, every candidate re-scored through the deployed pipeline,
and reported as a lower bound. Training runs with `python -m dr_evt_market train` on
synthetic windows or on windows logged by a run. The network below is untrained; it
shows only that the plug fits.

In [30]:
from dr_evt_market.mechanisms import regretformer
show(regretformer.RegretFormer.decide)
sessions = {p.name: m.InProcessPlatform(p.name, p.total_nodes, WORK / "rf" / p.name) for p in specs}
report_rf = m.Controller(sessions, platforms, regretformer.RegretFormer(None, seed=0), jobs, window_s=60, seed=0).run()
print("untrained RegretFormer: routed", len(report_rf.routed), "| net value", round(sum(w.welfare for w in report_rf.windows), 1),
      "| charges", round(sum(w.revenue for w in report_rf.windows), 1))
print("VCG on the same stream: routed", len(report.routed), "| net value", round(sum(w.welfare for w in report.windows), 1),
      "| charges", round(sum(w.revenue for w in report.windows), 1))

```python
    def decide(self, window: Window) -> list[Decision]:
        """Round one network pass into feasible, individually rational decisions."""
        if not window.jobs:
            return []
        structure = self._windows.structure_from_window(window)
        batch = self._windows.WindowBatch((structure,), self.device)
        assignments, payments = self._deploy.deployed_outcome(
            self.net, batch, batch.true_values
        )
        decisions = []
        for job_index, candidate_index in enumerate(assignments[0]):
            if candidate_index < 0:
                continue
            job = window.jobs[job_index]
            placement = window.candidates[job.job_id][candidate_index]
            if placement.value_credits < placement.cost_credits:
                continue
            charge = min(
                placement.value_credits,
                max(placement.cost_credits, float(payments[0, job_index])),
            )
            decisions.append(Decision(job.job_id, placement, charge))
        return decisions
```

untrained RegretFormer: routed 13 | net value 41.5 | charges 41.3
VCG on the same stream: routed 13 | net value 44.6 | charges 35.4


## Where to read more

`docs/api/MARKET_PYTHON_API.md` documents the contract, both adapters, the market's
records, VCG, the controller, the input and output files, the command line and the
learned mechanism. `tests/run_market_tests.sh` runs the package's test suite, and
`python/examples/market/` holds the two input files used here.